# Notebook 03 — Play with Graph Features

Build a bipartite user-merchant graph, compute degree / clustering / community /
RWSE features, append them to transactions, train the graph-aware model, and
compare it against the minimal baseline.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from phase01_lightgbm_baseline import GraphFeatureExtractor, LightGBMWrapper
from phase01_lightgbm_baseline.comparison import compute_comparison_report
from src.utils.features import BASELINE_FEATURES

rng = np.random.default_rng(0)
n = 3000
cards = rng.integers(0, 200, size=n)
products = rng.choice(list("WCHRS"), size=n)
fraud = (rng.random(n) < 0.04).astype(int)

df = pd.DataFrame(
    {
        "TransactionID": np.arange(n),
        "card1": cards,
        "ProductCD": products,
        "TransactionAmt": rng.lognormal(3, 1, size=n),
        "TransactionDT": np.sort(rng.integers(0, 5_000_000, size=n)),
        "isFraud": fraud,
    }
)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["isFraud"], random_state=42)
print(df.shape, train_df.shape, test_df.shape)

In [ ]:
extractor = GraphFeatureExtractor()
# Build the graph from the training transactions only (no leakage).
graph = extractor.build_bipartite_graph(train_df)
table = extractor.compute_feature_table(graph)
print(graph.number_of_nodes(), graph.number_of_edges())
print(list(table.columns))
table.head()

In [ ]:
from phase01_lightgbm_baseline import split_and_prepare, train_graph_aware

graph_cols = extractor.graph_feature_columns(table)
print(f"graph feature count: {len(graph_cols)}")

k = max(1, int(np.ceil(0.01 * len(test_df))))

# Leakage-free baseline features shared by both paths (fit on the train split).
train_feat, val_feat, test_feat, params = split_and_prepare((train_df, test_df))

minimal = LightGBMWrapper()
minimal_metrics = minimal.full_pipeline(
    train_feat[BASELINE_FEATURES].to_numpy(float), train_feat["isFraud"].to_numpy(int),
    val_feat[BASELINE_FEATURES].to_numpy(float), val_feat["isFraud"].to_numpy(int),
    test_feat[BASELINE_FEATURES].to_numpy(float), test_feat["isFraud"].to_numpy(int),
    k=k,
)

# Graph-aware model via the canonical pipeline (graph rebuilt on the train split only).
graph_model, graph_metrics = train_graph_aware(
    (train_df, test_df), extractor, k=k, wrapper=LightGBMWrapper()
)

deltas = compute_comparison_report(minimal_metrics, graph_metrics)
print(f"minimal PR-AUC: {minimal_metrics['pr_auc']:.4f}")
print(f"graph   PR-AUC: {graph_metrics['pr_auc']:.4f}")
print(f"delta PR-AUC  : {deltas['delta_pr_auc']:+.4f}")

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

users = table[table["node_type"] == "user"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(users["degree"], bins=30, color="steelblue")
axes[0].set(xlabel="Degree", ylabel="Users", title="User degree distribution")
axes[1].hist(users["community_label"], bins=max(2, users["community_label"].nunique()), color="tab:purple")
axes[1].set(xlabel="Community", ylabel="Users", title="Community size distribution")
fig.tight_layout()
fig.savefig("nb03_graph_features.png", dpi=120)
print("saved nb03_graph_features.png")

Graph features add relational context; whether they improve PR-AUC is exactly what the stop decision measures.